In [48]:
import os
import pandas as pd
import numpy as np
import joblib
import optuna
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer, accuracy_score, confusion_matrix, classification_report, precision_recall_curve

fraud_scorer = make_scorer(f1_score, pos_label=1)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [49]:
df_train = pd.read_csv(r'/home/frank/MACHINE LEARNING WSL/InsurAI-Agent/artifacts/train data.csv')
df_train.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Nov,1,Friday,Honda,Urban,Monday,Nov,1,Female,Married,...,6 years,36 to 40,No,No,External,more than 5,no change,1 vehicle,1996,Collision
1,Dec,5,Friday,Chevrolet,Urban,Monday,Jan,1,Male,Married,...,more than 7,51 to 65,No,No,External,none,no change,1 vehicle,1994,Collision
2,Jan,1,Tuesday,Dodge,Urban,Wednesday,Feb,1,Male,Married,...,more than 7,51 to 65,No,No,External,3 to 5,no change,1 vehicle,1994,Collision
3,Feb,1,Friday,Toyota,Urban,Wednesday,Feb,2,Male,Married,...,7 years,31 to 35,No,No,External,none,no change,1 vehicle,1996,All Perils
4,Jun,5,Tuesday,Pontiac,Urban,Tuesday,Jun,5,Male,Married,...,4 years,31 to 35,No,No,External,more than 5,no change,1 vehicle,1994,Collision


In [50]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 10794 entries, 0 to 10793
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Month                 10794 non-null  str  
 1   WeekOfMonth           10794 non-null  int64
 2   DayOfWeek             10794 non-null  str  
 3   Make                  10794 non-null  str  
 4   AccidentArea          10794 non-null  str  
 5   DayOfWeekClaimed      10794 non-null  str  
 6   MonthClaimed          10794 non-null  str  
 7   WeekOfMonthClaimed    10794 non-null  int64
 8   Sex                   10794 non-null  str  
 9   MaritalStatus         10794 non-null  str  
 10  Age                   10794 non-null  int64
 11  Fault                 10794 non-null  str  
 12  VehicleCategory       10794 non-null  str  
 13  VehiclePrice          10794 non-null  str  
 14  Deductible            10794 non-null  int64
 15  DriverRating          10794 non-null  int64
 16  Days_Policy_Acc

In [51]:
df_train_target = pd.read_csv(r'/home/frank/MACHINE LEARNING WSL/InsurAI-Agent/artifacts/train target data.csv')
df_train_target.head()

,FraudFound_P
0,0
1,0
2,0
3,0
4,0


In [52]:
x = df_train
y = df_train_target


In [53]:
x_train, x_validate, y_train, y_validate = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

In [54]:
# lgbm_model = lgb.LGBMClassifier(scale_pos_weight=15)

In [55]:
# master_pipeline_lgb = Pipeline([        
#     ('classifier', lgbm_model)               
# ])

In [56]:
# master_pipeline_lgb.fit(x_train, y_train)

In [57]:
# y_pred_lgb = master_pipeline_lgb.predict(x_validate)

In [58]:
# CR_lgb = classification_report(y_validate, y_pred_lgb, output_dict=True)
# CR_lgb = pd.DataFrame(CR_xgb)
# CR_lgb

In [ ]:
cat_feature_indices = [i for i, dtype in enumerate(x_train.dtypes)
                       if str(dtype) == 'str' or str(dtype) == 'object']
print(f"Categorical columns found: {len(cat_feature_indices)}")
print(f"Columns: {x_train.columns[cat_feature_indices].tolist()}")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

Categorical columns found: 23
Columns: ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy']


In [ ]:
def catboost_objective(trial):
    fraud_weight = trial.suggest_float('fraud_weight', 3, 20)

    params = {
        'iterations':          trial.suggest_int('iterations', 200, 600),
        'depth':               trial.suggest_int('depth', 3, 8),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength':     trial.suggest_float('random_strength', 0, 2),
        'border_count':        trial.suggest_int('border_count', 32, 128),
        'class_weights':       [1, fraud_weight],
        'eval_metric':         'F1',
        'random_seed':         42,
        'verbose':             False,
        'thread_count':        1,
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []

    x = x_train.reset_index(drop=True)
    y = y_train.values.ravel()
 
    for train_idx, val_idx in skf.split(x, y):
        x_fold_train, x_fold_val = x.iloc[train_idx], x.iloc[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]

        model = CatBoostClassifier(
            **params,
            cat_features=cat_feature_indices if cat_feature_indices else None
        )
        model.fit(x_fold_train, y_fold_train)

        y_pred = model.predict(x_fold_val)
        score = f1_score(y_fold_val, y_pred, pos_label=1)
        fold_scores.append(score)

    return np.mean(fold_scores)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
cat_study2 = optuna.create_study(direction='maximize')
cat_study2.optimize(catboost_objective, n_trials=50, show_progress_bar=True)

print("Best F1:", cat_study2.best_value)
print("Best params:", cat_study2.best_params)

Best trial: 6. Best value: 0.262416: 100%|██████████| 50/50 [13:30<00:00, 16.22s/it]

Best F1: 0.2624157039050656
Best params: {'fraud_weight': 12.081006947176506, 'iterations': 243, 'depth': 5, 'learning_rate': 0.05441432613946479, 'l2_leaf_reg': 5.487036397806115, 'bagging_temperature': 0.05063784785748893, 'random_strength': 0.06645333815897692, 'border_count': 52}


In [ ]:
best_p = cat_study2.best_params.copy()
fraud_weight_final = best_p.pop('fraud_weight')
best_p.update({
    'class_weights': [1, fraud_weight_final],
    'eval_metric':   'F1',
    'random_seed':   42,
    'verbose':       False,
    'thread_count':  -1,
})

final_cat = CatBoostClassifier(
    **best_p,
    cat_features=cat_feature_indices if cat_feature_indices else None
)
final_cat.fit(x_train, y_train.values.ravel())
print("Final model trained.")

Final model trained.


In [ ]:
y_prob = final_cat.predict_proba(x_validate)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_validate.values.ravel(), y_prob)

beta = 1.5
fbeta_scores = (1 + beta**2) * (precisions[:-1] * recalls[:-1]) / ((beta**2 * precisions[:-1]) + recalls[:-1] + 1e-9)

best_idx = np.argmax(fbeta_scores)
best_threshold = thresholds[best_idx]

print(f"\nBest threshold (by F2): {best_threshold:.4f}  "
      f"precision={precisions[best_idx]:.3f}  "
      f"recall={recalls[best_idx]:.3f}  "
      f"F2={fbeta_scores[best_idx]:.3f}")


Best threshold (by F2): 0.5603  precision=0.202  recall=0.619  F2=0.378


In [ ]:
best_threshold = 0.52

y_prob = final_cat.predict_proba(x_validate)[:, 1]

y_pred_final = (y_prob >= best_threshold).astype(int)

print(f"\n{'='*55}")
print(f"CATBOOST FINAL — LOCKED THRESHOLD {best_threshold}")
print('='*55)
print(classification_report(y_validate.values.ravel(), y_pred_final, zero_division=0))

cm = confusion_matrix(y_validate.values.ravel(), y_pred_final)
print("Confusion Matrix:")
display(pd.DataFrame(cm,
                   index=['Actual Legit', 'Actual Fraud'],
                   columns=['Predicted Legit', 'Predicted Fraud']))


CATBOOST FINAL — LOCKED THRESHOLD 0.52
              precision    recall  f1-score   support

           0       0.98      0.79      0.88      3045
           1       0.17      0.69      0.28       194

    accuracy                           0.79      3239
   macro avg       0.58      0.74      0.58      3239
weighted avg       0.93      0.79      0.84      3239

Confusion Matrix:


,Predicted Legit,Predicted Fraud
Actual Legit,2417,628
Actual Fraud,61,133


In [66]:
# ── STEP 7: Final Evaluation ────────────────────────────────────────
print(f"\n{'='*55}")
print(f"CATBOOST FINAL — THRESHOLD {best_threshold:.4f}")
print('='*55)
y_pred_final = (y_prob >= best_threshold).astype(int)
print(classification_report(y_validate.values.ravel(), y_pred_final, zero_division=0))
cm = confusion_matrix(y_validate.values.ravel(), y_pred_final)
print("Confusion Matrix:")
print(pd.DataFrame(cm,
                   index=['Actual Legit', 'Actual Fraud'],
                   columns=['Predicted Legit', 'Predicted Fraud']))




CATBOOST FINAL — THRESHOLD 0.5200
              precision    recall  f1-score   support

           0       0.98      0.79      0.88      3045
           1       0.17      0.69      0.28       194

    accuracy                           0.79      3239
   macro avg       0.58      0.74      0.58      3239
weighted avg       0.93      0.79      0.84      3239

Confusion Matrix:
              Predicted Legit  Predicted Fraud
Actual Legit             2417              628
Actual Fraud               61              133


In [67]:
# ── STEP 8: Feature Importance ──────────────────────────────────────
feat_imp = pd.DataFrame({
    'feature':    x_train.columns,
    'importance': final_cat.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 15 features:")
print(feat_imp.head(15).to_string(index=False))




Top 15 features:
            feature  importance
              Fault   36.949021
    VehicleCategory   10.655402
         BasePolicy    7.648625
                Age    4.688430
   DayOfWeekClaimed    4.083729
               Year    3.928009
              Month    3.595325
          DayOfWeek    3.113584
               Make    2.866503
       MonthClaimed    2.446690
         Deductible    2.150578
  AgeOfPolicyHolder    2.140750
       AgeOfVehicle    2.128062
AddressChange_Claim    2.037441
                Sex    1.674019


In [69]:
df_test = pd.read_csv(r'/home/frank/MACHINE LEARNING WSL/InsurAI-Agent/artifacts/test data.csv')
df_test.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Oct,2,Tuesday,Pontiac,Urban,Monday,Oct,3,Male,Married,...,7 years,41 to 50,No,No,External,3 to 5,no change,1 vehicle,1996,Liability
1,Mar,2,Monday,Honda,Urban,Monday,Mar,2,Female,Divorced,...,7 years,36 to 40,No,No,External,none,no change,1 vehicle,1996,Liability
2,Apr,4,Sunday,Pontiac,Urban,Tuesday,May,1,Female,Single,...,more than 7,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Dec,5,Friday,Accura,Urban,Monday,Jan,2,Male,Married,...,more than 7,36 to 40,No,No,External,1 to 2,no change,1 vehicle,1994,All Perils
4,Oct,3,Saturday,Pontiac,Urban,Monday,Oct,3,Male,Married,...,6 years,36 to 40,No,No,External,3 to 5,no change,1 vehicle,1994,Collision


In [70]:
df_test_target = pd.read_csv(r'/home/frank/MACHINE LEARNING WSL/InsurAI-Agent/artifacts/test target data.csv')
df_test_target.head()

,FraudFound_P
0,0
1,0
2,0
3,0
4,0


In [73]:
X_test = df_test
Y_test = df_test_target

In [74]:
Y_prob_test = final_cat.predict_proba(X_test)[:, 1]
Y_pred_test = (Y_prob_test >= best_threshold).astype(int)

In [75]:
if Y_test is not None:
    from sklearn.metrics import classification_report, confusion_matrix
    print(classification_report(Y_test.values.ravel(), Y_pred_test, zero_division=0))
    print(pd.DataFrame(
        confusion_matrix(Y_test.values.ravel(), Y_pred_test),
        index=['Actual Legit', 'Actual Fraud'],
        columns=['Predicted Legit', 'Predicted Fraud']
    ))
else:
    print("Predictions done — no labels to evaluate against")
    print(pd.Series(Y_pred_test).value_counts().rename({0: 'Legit', 1: 'Fraud'}))

              precision    recall  f1-score   support

           0       0.98      0.78      0.87      4349
           1       0.17      0.70      0.27       277

    accuracy                           0.78      4626
   macro avg       0.57      0.74      0.57      4626
weighted avg       0.93      0.78      0.83      4626

              Predicted Legit  Predicted Fraud
Actual Legit             3408              941
Actual Fraud               84              193


In [76]:
final_cat.save_model("catboost_fraud_model.cbm")